In [ ]:
# Install YOLO (Ultralytics) and Roboflow libraries quietly (-q)
!pip install ultralytics roboflow -q

# Import the YOLO library to verify installation
import ultralytics
ultralytics.checks()

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.1/112.6 GB disk)


In [ ]:
from google.colab import userdata

In [ ]:
import os
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API'))
project = rf.workspace("ammar-salah-ahmed-s-workspace").project("safety-rd-v1-rqwku")
version = project.version(1)
dataset = version.download("yolov11")

print(f"Dataset downloaded to: {dataset.location}")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to safety-RD-v1-1 in yolov11:: 100%|██████████| 33909/33909 [00:06<00:00, 5079.87it/s]


Dataset downloaded to: /content/safety-RD-v1-1


In [ ]:
import yaml

yaml_path = f"{dataset.location}/data.yaml"

# Read and print the YAML file to ensure paths and classes are correct
with open(yaml_path, 'r') as file:
    yaml_data = yaml.safe_load(file)

print("Classes in dataset:", yaml_data.get('names'))

Classes in dataset: ['Gloves', 'helmet', 'no-glove', 'no-helmet', 'no-vest', 'person', 'vest']


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project="Safety_Detection",
    name="yolov11_ppe_run1",
    device=0

)

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/safety-RD-v1-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11_ppe_run1-3, nbs=64,

In [ ]:
import os
import shutil
from google.colab import files

# Define the path to the directory containing the training results
results_dir = '/content/runs/detect/Safety_Detection/yolov11_ppe_run1-3'

# Define the name for the zip file
zip_filename = 'yolov11_ppe_training_results.zip'

# Create a zip archive of the results directory
shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', results_dir)

print(f"Successfully created {zip_filename} in the current directory.")

# Offer the zip file for download
files.download(zip_filename)

Successfully created yolov11_ppe_training_results.zip in the current directory.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
metrics = model.val()

print(f"Mean Average Precision (mAP50-95): {metrics.box.map}")

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,415,509 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1048.6±439.6 MB/s, size: 75.2 KB)
val: Scanning /content/safety-RD-v1-1/valid/labels.cache... 3390 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3390/3390 646.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 212/212 4.0it/s 52.4s
